# 2Day end 役職推定パイプライン実行ノートブック

**2日目開始時点でのデータを使用して役職を推定するパイプライン**

このノートブックは`run_train_pipeline.py`の構造を参考にしながら、2日目開始シナリオに対応させたものです。

このノートブックで行うこと:
- 実行環境とパスの設定
- 既存モジュールの import と自動リロード
- 2日目開始時のデータ準備（day==1フィルタ、脱落者データ除外）
- 役職割り当てロジックを用いた訓練・評価
- 特徴量重要度と SHAP による解釈可能性分析
- CSV/画像の保存と結果確認

注意:
- このパイプラインは **day==1** のデータのみを使用します
- 2日目の脱落者情報（exec_id, attack_id）を基にした制約を考慮します

## 1. Notebookで使う実行環境・パス設定

Python バージョン確認、ライブラリ import、プロジェクトルートと `sys.path` を設定します。

In [1]:
import os
import sys
import json
import time
import traceback
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python: {sys.version}")

# notebooks/ から見たプロジェクトルート
PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / "src"
MODELS_DIR = PROJECT_ROOT / "models"
CONFIG_PATH = PROJECT_ROOT / "config" / "training_config.json"
DATA_DIR = PROJECT_ROOT / "data"

# パスを sys.path に追加
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SRC_PATH: {SRC_PATH}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"CONFIG_PATH exists: {CONFIG_PATH}")
print(f"DATA_DIR: {DATA_DIR}")

Python: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
PROJECT_ROOT: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub
SRC_PATH: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\src
MODELS_DIR: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\models
CONFIG_PATH exists: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\config\training_config.json
DATA_DIR: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\data


## 2. 既存 `.py` のインポートと自動リロード設定

モジュール更新を即時反映するため、autoreload を有効化します。

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# プロジェクトルートとsrcディレクトリの絶対パスをsys.pathに追加
PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / "src"
if str(PROJECT_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.resolve()))
if str(SRC_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_PATH.resolve()))

from src.pipelines.training_pipeline import load_training_config, get_data_paths, get_models_dir
print("Module import complete.")

Module import complete.


C:\Users\takic\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. 設定読み込みと2Day Start用パラメータ管理

設定ファイルを読み込み、day_filter=1（1日目データ）に固定して2日目開始時のシナリオに対応させます。

In [3]:
RUN_OPTIONS = {
    "run_training": True,
    "top_k_features": 10,
    "show_only_required_artifacts": True,
    "day2_flag": True,  # 2日目脱落者制約を有効
}

config = load_training_config(CONFIG_PATH)

# 2Day Start用に設定をオーバーライド
override = {
    "n_trials": 30,  # 探索回数を10回に固定（短時間実行用）
}

# 見た目と実際の挙動を一致させるため、config側にも反映
config["n_trials"] = int(override["n_trials"])

print("="*70)
print("2DAY START パイプライン設定")
print("="*70)
print("\nRun options:")
print(json.dumps(RUN_OPTIONS, ensure_ascii=False, indent=2))
print("\nTraining config (effective):")
print(json.dumps(config, ensure_ascii=False, indent=2))
print("\n2Day Start override:")
print(json.dumps(override, ensure_ascii=False, indent=2))
print(f"\nEffective n_trials: {config['n_trials']}")
print("\nFeature policy:")
if bool(config.get("lang_feature", False)):
    print("- lang_feature=True ですが、話者推定モデル/話者推定確率特徴は無効です。")
else:
    print("- 話者推定モデル/話者推定確率特徴は使用しません。")

2DAY START パイプライン設定

Run options:
{
  "run_training": true,
  "top_k_features": 10,
  "show_only_required_artifacts": true,
  "day2_flag": true
}

Training config (effective):
{
  "data_paths": [
    "data/2025spring/all_feature_table_2025sp17_with_talks.csv",
    "data/2025summer/all_feature_table_2025summer_with_talks2.csv"
  ],
  "data_paths_day2": [
    "data/2025spring/all_feature_table_2025sp17_day2_with_talks.csv",
    "data/2025summer/all_feature_table_2025summer_day2_with_talks.csv"
  ],
  "day_filter": 1,
  "lang_feature": false,
  "group_column": "source_file",
  "cv_folds": 5,
  "test_size": 0.2,
  "n_trials": 30,
  "leakage_drop_columns": [
    "True_Div_recepient_id_1",
    "True_Div_result_1",
    "True_Div_recepient_id_2",
    "True_Div_result_2",
    "target_total_votes",
    "alive"
  ]
}

2Day Start override:
{
  "n_trials": 30
}

Effective n_trials: 30

Feature policy:
- 話者推定モデル/話者推定確率特徴は使用しません。


## 4. 2Day vote用データ準備

day==1 のデータのみを読み込み、脱落者情報を基にした制約を準備します。

In [10]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import joblib

print("="*70)
print("2DAY START データ準備")
print("="*70)

# データパスの取得
data_paths = get_data_paths(config, data_type="day2")

# CSVの読み込みと day==1 フィルタ
dfs = []
for p in data_paths:
    p_path = Path(p)  # 文字列をPathオブジェクトに変換
    df_raw = pd.read_csv(p)
    dataset_name = p_path.parent.name 
    df_raw['dataset_tag'] = dataset_name
    dfs.append(df_raw)

df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"\n✓ Combined shape: {df.shape}")

# attack_id が NaN のサンプルを除外
df_filtered = df.dropna(subset=["attack_id"]).copy()
print(f"✓ After dropping NaN attack_id: {df_filtered.shape}")

# SEER で True_Div_result_1 が NaN のファイルを除外
if "True_Div_result_1" in df_filtered.columns:
    del_paths = df_filtered[(df_filtered["role"]=="SEER") & (df_filtered["True_Div_result_1"].isna())]["source_file"].unique()
    if len(del_paths) > 0:
        print(f"  Excluding {len(del_paths)} files with missing True_Div_result_1 for SEER")
        df_filtered = df_filtered[~df_filtered["source_file"].isin(del_paths)]

# 目的変数のエンコード
label_encoder = LabelEncoder()
df_filtered['role_encoded'] = label_encoder.fit_transform(df_filtered['role'].astype(str))
y = df_filtered['role_encoded'].values

# メタ情報の抽出
meta = {
    'source_file': df_filtered['source_file'].values,
    'dataset_tag': df_filtered['dataset_tag'].values if 'dataset_tag' in df_filtered.columns else np.array(["unknown"] * len(df_filtered)),
    'div_result1': df_filtered['True_Div_result_1'].values if 'True_Div_result_1' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'div_id1': df_filtered['True_Div_recepient_id_1'].values if 'True_Div_recepient_id_1' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'div_result2': df_filtered['True_Div_result_2'].values if 'True_Div_result_2' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'div_id2': df_filtered['True_Div_recepient_id_2'].values if 'True_Div_recepient_id_2' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'exec_id': df_filtered['exec_id'].values if 'exec_id' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'attack_id': df_filtered['attack_id'].values if 'attack_id' in df_filtered.columns else np.full(len(df_filtered), np.nan),
}

# 特徴量行列の作成
# = df_filtered['combined_text'].astype(str).values
agent_names = df_filtered['agent_name'].astype(str).values

base_drop_cols = [
    'id', 'role', 'source_file', 'dataset_tag', 'day', 'role_encoded', 'role_encoded',
    'Est_id_Fact_role', 'Est_id_Est_roles', 'character_name',
    'agent_name', 'combined_text', 'seer_co_order'
]
meta_raw_cols = [
    'True_Div_result_1', 'True_Div_recepient_id_1',
    'True_Div_result_2', 'True_Div_recepient_id_2',
    'exec_id', 'attack_id'
]
day2_col = []
id_like_cols = [c for c in df_filtered.columns if c.endswith('_id')]
flag_like_cols = [c for c in df_filtered.columns if c.endswith('_flag')]

all_drop_cols = list(set(base_drop_cols + meta_raw_cols + id_like_cols + flag_like_cols + day2_col))
X_df = df_filtered.drop(columns=all_drop_cols, errors='ignore')
final_features = X_df.columns.tolist()
X = X_df.apply(pd.to_numeric, errors='coerce').fillna(0).values

print(f"\n✓ Data shapes:")
print(f"  X: {X.shape}")
print(f"  y: {y.shape}")
print(f"  Roles: {label_encoder.classes_}")
print(f"  Games: {len(np.unique(meta['source_file']))}")
print("  Dataset ratio:")
display(pd.Series(meta['dataset_tag']).value_counts().rename_axis('dataset').reset_index(name='samples'))

2DAY START データ準備
✓ Data files selected:
  - all_feature_table_2025sp17_day2_with_talks.csv
  - all_feature_table_2025summer_day2_with_talks.csv

✓ Combined shape: (1125, 450)
✓ After dropping NaN attack_id: (745, 450)
  Excluding 7 files with missing True_Div_result_1 for SEER

✓ Data shapes:
  X: (710, 420)
  y: (710,)
  Roles: ['POSSESSED' 'SEER' 'VILLAGER' 'WEREWOLF']
  Games: 142
  Dataset ratio:


,dataset,samples
0,2025summer,375
1,2025spring,335


## 5. 役職割り当て関数（制約付き予測）

SEER視点、WEREWOLF視点、VILLAGER視点などの異なる視点から役職を割り当てます。
脱落者（exec_id, attack_id）を基にした制約を組み込みます。

In [8]:
import itertools
import torch

def assign_roles_for_non_seer(
    logits, y_batch, role_counts, role_names, label_encoder,
    fixed_self_role_name, exec_id_batch, attack_id_batch, day2_flag=True, debug=False):
    """
    非SEER視点（VILLAGER/POSSESSED/WEREWOLF）での役職割り当て
    脱落者を人狼ではなく設定
    """
    role_names = list(role_names)
    if isinstance(logits, torch.Tensor): 
        logits = logits.detach().cpu().numpy()
    if isinstance(y_batch, torch.Tensor): 
        y_batch = y_batch.detach().cpu().numpy()
    
    num_players = logits.shape[0]
    if num_players != 5:
        return np.array([]), np.array([])
    
    pred_list, y_list = [], []

    for self_index in range(num_players):
        self_role = y_batch[self_index]
        self_role_name = label_encoder.inverse_transform([self_role])[0]
        
        if self_role_name != fixed_self_role_name:
            continue

        if day2_flag and (self_index + 1 in exec_id_batch or self_index + 1 in attack_id_batch):
            if debug:
                print(f"skip id:{self_index + 1} exec or attack")
            continue

        other_indices = [i for i in range(num_players) if i != self_index]
        logits_others = logits[other_indices]
        y_others = y_batch[other_indices]
        
        non_werewolf_indices_in_others = []
        if day2_flag:
            exec_id = exec_id_batch[self_index]
            attack_id = attack_id_batch[self_index]
            if np.isnan(attack_id) or np.isnan(exec_id): 
                continue
            attack_id_idx = int(attack_id) - 1
            exec_id_idx = int(exec_id) - 1
            if self_index == exec_id_idx or self_index == attack_id_idx: 
                continue
            if exec_id_idx in other_indices:
                non_werewolf_indices_in_others.append(other_indices.index(exec_id_idx))
            if attack_id_idx in other_indices:
                non_werewolf_indices_in_others.append(other_indices.index(attack_id_idx))
            non_werewolf_indices_in_others = list(set(non_werewolf_indices_in_others))
        
        reduced_counts = role_counts.copy()
        if reduced_counts.get(fixed_self_role_name, 0) == 0:
            continue
        reduced_counts[fixed_self_role_name] -= 1
        roles_pool = [role for role, count in reduced_counts.items() for _ in range(count)]
        if len(roles_pool) != 4: 
            continue
        
        best_perm = None
        best_score = -np.inf
        for perm in set(itertools.permutations(roles_pool)):
            if any(perm[idx] == "WEREWOLF" for idx in non_werewolf_indices_in_others):
                continue
            score = sum(np.log(logits_others[i][role_names.index(role)] + 1e-9) for i, role in enumerate(perm))
            if score > best_score:
                best_score, best_perm = score, perm
        
        if best_perm:
            pred_encoded = label_encoder.transform(list(best_perm))
            pred_list.append(pred_encoded)
            y_list.append(y_others)

    return (np.concatenate(pred_list) if pred_list else np.array([]),
            np.concatenate(y_list) if y_list else np.array([]))


def assign_roles_for_seer_by_divination(
    logits, y_batch, role_counts, role_names, label_encoder,
    div_result_array1, div_id_array1, div_result_array2=None, div_id_array2=None,
    exec_id_batch=None, attack_id_batch=None, day2_flag=True, debug=False):
    """
    SEER視点での役職割り当て
    占い結果を基に黒（人狼）/白（人狼以外）を分類
    """
    role_names = list(role_names)
    if isinstance(logits, torch.Tensor): 
        logits = logits.detach().cpu().numpy()
    if isinstance(y_batch, torch.Tensor): 
        y_batch = y_batch.detach().cpu().numpy()
    
    num_players = logits.shape[0]
    if num_players != 5:
        empty_result = (np.array([]), np.array([]))
        return {"black": empty_result, "white": empty_result}

    pred_list_black, y_list_black = [], []
    pred_list_white, y_list_white = [], []

    for self_index in range(num_players):
        self_role_name = label_encoder.inverse_transform([y_batch[self_index]])[0]

        if self_role_name != "SEER":
            continue

        if day2_flag and (self_index+1 in exec_id_batch or self_index+1 in attack_id_batch):
            continue

        other_indices = [i for i in range(num_players) if i != self_index]
        logits_others = logits[other_indices]
        y_others = y_batch[other_indices]

        non_werewolf_indices_in_others = []
        if day2_flag and exec_id_batch is not None and attack_id_batch is not None:
            exec_id = exec_id_batch[self_index]
            attack_id = attack_id_batch[self_index]
            if not (np.isnan(exec_id) or np.isnan(attack_id)):
                exec_id_idx = int(exec_id) - 1
                attack_id_idx = int(attack_id) - 1
                if exec_id_idx in other_indices:
                    non_werewolf_indices_in_others.append(other_indices.index(exec_id_idx))
                if attack_id_idx in other_indices:
                    non_werewolf_indices_in_others.append(other_indices.index(attack_id_idx))
        
        div_constraints = {}
        all_div_results = [(div_result_array1, div_id_array1)]
        if day2_flag and div_result_array2 is not None and div_id_array2 is not None:
            all_div_results.append((div_result_array2, div_id_array2))

        first_div_result_for_classification = None

        for i, (div_result_array, div_id_array) in enumerate(all_div_results):
            if np.isnan(div_id_array[self_index]): 
                continue
            
            div_id = int(div_id_array[self_index]) - 1
            if div_id not in other_indices: 
                continue
            
            div_target_index_in_others = other_indices.index(div_id)
            div_result_str = str(div_result_array[self_index]).strip()

            if div_result_str in ["WEREWOLF", "人狼", "黒"]:
                div_constraints[div_target_index_in_others] = "WEREWOLF"
                if i == 0: 
                    first_div_result_for_classification = "WEREWOLF"
            elif div_result_str in ["HUMAN", "白", "not(人狼)"]:
                div_constraints[div_target_index_in_others] = "HUMAN"
                if i == 0: 
                    first_div_result_for_classification = "HUMAN"
        
        if first_div_result_for_classification is None:
            continue

        reduced_counts = role_counts.copy()
        if reduced_counts.get("SEER", 0) == 0: 
            continue
        reduced_counts["SEER"] -= 1
        roles_pool = [role for role, count in reduced_counts.items() for _ in range(count)]
        if len(roles_pool) != 4: 
            continue

        best_score, best_perm = -np.inf, None

        for perm in set(itertools.permutations(roles_pool)):
            is_valid = True
            
            if any(perm[idx] == "WEREWOLF" for idx in non_werewolf_indices_in_others):
                continue
            
            for target_idx, result in div_constraints.items():
                if result == "WEREWOLF" and perm[target_idx] != "WEREWOLF":
                    is_valid = False; break
                if result == "HUMAN" and perm[target_idx] == "WEREWOLF":
                    is_valid = False; break
            if not is_valid: 
                continue

            score = sum(np.log(logits_others[i][role_names.index(role)] + 1e-9) for i, role in enumerate(perm))
            if score > best_score:
                best_score, best_perm = score, perm

        if best_perm:
            pred_encoded = label_encoder.transform(list(best_perm))
            if first_div_result_for_classification == "WEREWOLF":
                pred_list_black.append(pred_encoded)
                y_list_black.append(y_others)
            elif first_div_result_for_classification == "HUMAN":
                pred_list_white.append(pred_encoded)
                y_list_white.append(y_others)

    result = {
        "black": (np.concatenate(pred_list_black) if pred_list_black else np.array([]),
                  np.concatenate(y_list_black) if y_list_black else np.array([])),
        "white": (np.concatenate(pred_list_white) if pred_list_white else np.array([]),
                  np.concatenate(y_list_white) if y_list_white else np.array([])),
    }
    return result

print("✓ 役職割り当て関数が定義されました")

✓ 役職割り当て関数が定義されました


## 6. モデルの訓練・最適化（StratifiedGroupKFold with CV）

4つの視点モデル（SEER, WEREWOLF, VILLAGER, POSSESSED）をOptuna で最適化し、
StratifiedGroupKFold によるクロスバリデーションで評価します。

In [11]:
import xgboost as xgb
import optuna
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter

print("="*70)
print("2DAY START モデル訓練 (Optuna + Balanced Fold Split)")
print("="*70)

t0 = time.time()
predictor = None
run_status = {"ok": False, "error": None, "elapsed_sec": None}

try:
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    weight_dict = dict(zip(classes, weights))

    groups = np.array([f"{d}::{s}" for d, s in zip(meta['dataset_tag'], meta['source_file'])])
    role_counts = {'POSSESSED': 1, 'SEER': 1, 'VILLAGER': 2, 'WEREWOLF': 1}
    class_names = list(label_encoder.classes_)

    n_trials = int(override.get("n_trials", 100))
    n_splits = 5
    fold_seed = 42
    fold_search_iter = 80

    print(f"\n✓ Data prepared for training:")
    print(f"  Samples: {len(X)}")
    print(f"  Features: {len(final_features)}")
    print(f"  Classes: {class_names}")
    print(f"  CV folds: {n_splits}")
    print(f"  Optuna trials: {n_trials}")

    # ------------------------------
    # Balanced fold split (game-wise)
    # 目的:
    #  1) 連結元データ(dataset_tag)比率をfold間で均等化
    #  2) 各agentの分布をfold間で均等化
    # ------------------------------
    group_keys = np.array(groups)
    unique_games = np.unique(group_keys)
    dataset_tags = np.array(meta['dataset_tag'])

    # 1ゲームごとのインデックスと集計情報
    game_indices = {}
    game_agent_counter = {}
    game_dataset_counter = {}
    game_pair_counter = {}
    all_agents = set()
    all_datasets = set()
    all_pairs = set()

    for g in unique_games:
        idx = np.where(group_keys == g)[0]
        game_indices[g] = idx

        agent_cnt = Counter(agent_names[i] for i in idx)
        ds_cnt = Counter(dataset_tags[i] for i in idx)
        pair_cnt = Counter((agent_names[i], int(y[i])) for i in idx)

        game_agent_counter[g] = agent_cnt
        game_dataset_counter[g] = ds_cnt
        game_pair_counter[g] = pair_cnt

        all_agents.update(agent_cnt.keys())
        all_datasets.update(ds_cnt.keys())
        all_pairs.update(pair_cnt.keys())

    all_agents = sorted(all_agents)
    all_datasets = sorted(all_datasets)
    all_pairs = sorted(all_pairs)

    def ratio_mse_after_add(fold_counters, add_counter, keys, target_fold):
        # 各キーについて、foldごとの比率が1/n_splitsに近いほど小さいスコア
        total_mse = 0.0
        eps = 1e-12
        ideal = 1.0 / n_splits

        for key in keys:
            vals = []
            for f in range(n_splits):
                base = fold_counters[f].get(key, 0)
                vals.append(base + (add_counter.get(key, 0) if f == target_fold else 0))
            vals = np.array(vals, dtype=float)
            s = vals.sum()
            if s <= eps:
                continue
            ratios = vals / s
            total_mse += float(np.mean((ratios - ideal) ** 2))

        return total_mse / max(len(keys), 1)

    def variance_after_add(fold_counts, add_counter, keys, target_fold):
        total_var = 0.0
        for key in keys:
            vals = []
            for f in range(n_splits):
                base = fold_counts[f].get(key, 0)
                vals.append(base + (add_counter.get(key, 0) if f == target_fold else 0))
            total_var += float(np.var(vals))
        return total_var / max(len(keys), 1)

    def greedy_assign_once(seed: int):
        rng = np.random.default_rng(seed)
        game_order = list(unique_games)
        rng.shuffle(game_order)

        fold_agent_counts = [Counter() for _ in range(n_splits)]
        fold_dataset_counts = [Counter() for _ in range(n_splits)]
        fold_pair_counts = [Counter() for _ in range(n_splits)]
        fold_game_counts = [0 for _ in range(n_splits)]
        game_to_fold = {}

        # スコア: dataset比率 + agent比率 + agent-role分布 + foldサイズ
        # dataset比率とagent比率を強めに最適化
        w_dataset = 1.4
        w_agent = 1.2
        w_pair = 0.35
        w_size = 0.2

        def score_fold_with_candidate(fold_idx, game_id):
            ds_score = ratio_mse_after_add(
                fold_dataset_counts,
                game_dataset_counter[game_id],
                all_datasets,
                fold_idx,
            )
            agent_score = ratio_mse_after_add(
                fold_agent_counts,
                game_agent_counter[game_id],
                all_agents,
                fold_idx,
            )
            pair_score = variance_after_add(
                fold_pair_counts,
                game_pair_counter[game_id],
                all_pairs,
                fold_idx,
            )

            size_vec = np.array(fold_game_counts, dtype=float)
            size_vec[fold_idx] += 1.0
            size_imbalance = float(np.var(size_vec))

            return (
                w_dataset * ds_score
                + w_agent * agent_score
                + w_pair * pair_score
                + w_size * size_imbalance
            )

        for g in game_order:
            best_fold = 0
            best_score = np.inf

            for f in range(n_splits):
                s = score_fold_with_candidate(f, g)
                if s < best_score:
                    best_score = s
                    best_fold = f

            game_to_fold[g] = best_fold
            fold_game_counts[best_fold] += 1
            fold_agent_counts[best_fold].update(game_agent_counter[g])
            fold_dataset_counts[best_fold].update(game_dataset_counter[g])
            fold_pair_counts[best_fold].update(game_pair_counter[g])

        final_score = 0.0
        final_score += w_dataset * ratio_mse_after_add(fold_dataset_counts, Counter(), all_datasets, 0)
        final_score += w_agent * ratio_mse_after_add(fold_agent_counts, Counter(), all_agents, 0)
        final_score += w_pair * variance_after_add(fold_pair_counts, Counter(), all_pairs, 0)
        final_score += w_size * float(np.var(np.array(fold_game_counts, dtype=float)))

        return game_to_fold, fold_pair_counts, fold_game_counts, final_score

    best = None
    best_score = np.inf
    for k in range(fold_search_iter):
        out = greedy_assign_once(seed=fold_seed + k)
        if out[3] < best_score:
            best = out
            best_score = out[3]

    game_to_fold, fold_pair_counts, fold_game_counts, _ = best

    fold_id_per_row = np.array([game_to_fold[g] for g in group_keys], dtype=int)

    split_indices = []
    for fold in range(n_splits):
        val_idx = np.where(fold_id_per_row == fold)[0]
        train_idx = np.where(fold_id_per_row != fold)[0]
        split_indices.append((train_idx, val_idx))

    # 分割サマリー（見やすい表）
    fold_summary_rows = []
    for fold in range(n_splits):
        val_idx = np.where(fold_id_per_row == fold)[0]
        fold_summary_rows.append(
            {
                "fold": fold + 1,
                "n_samples": int(len(val_idx)),
                "n_games": int(len(np.unique(group_keys[val_idx]))),
            }
        )
    fold_summary_df = pd.DataFrame(fold_summary_rows)

    # fold x role 分布
    role_dist_rows = []
    for fold in range(n_splits):
        val_idx = np.where(fold_id_per_row == fold)[0]
        y_fold = y[val_idx]
        for r in np.unique(y):
            role_dist_rows.append(
                {
                    "fold": fold + 1,
                    "role": label_encoder.inverse_transform([r])[0],
                    "count": int(np.sum(y_fold == r)),
                }
            )
    fold_role_df = pd.DataFrame(role_dist_rows)

    # fold x dataset 分布
    dataset_dist_rows = []
    for fold in range(n_splits):
        val_idx = np.where(fold_id_per_row == fold)[0]
        ds_vals = dataset_tags[val_idx]
        for ds in sorted(np.unique(dataset_tags)):
            dataset_dist_rows.append(
                {
                    "fold": fold + 1,
                    "dataset": str(ds),
                    "count": int(np.sum(ds_vals == ds)),
                }
            )
    fold_dataset_df = pd.DataFrame(dataset_dist_rows)

    # fold x agent 分布（上位表示）
    agent_dist_rows = []
    for fold in range(n_splits):
        val_idx = np.where(fold_id_per_row == fold)[0]
        ag_vals = agent_names[val_idx]
        for ag in sorted(np.unique(agent_names)):
            agent_dist_rows.append(
                {
                    "fold": fold + 1,
                    "agent": str(ag),
                    "count": int(np.sum(ag_vals == ag)),
                }
            )
    fold_agent_df = pd.DataFrame(agent_dist_rows)

    print("\nBalanced fold summary:")
    display(fold_summary_df)
    print("Fold x Dataset distribution:")
    display(fold_dataset_df.pivot(index="fold", columns="dataset", values="count").fillna(0).astype(int))
    print("Fold x Role distribution:")
    display(fold_role_df.pivot(index="fold", columns="role", values="count").fillna(0).astype(int))
    print("Fold x Agent distribution (first 20 agents):")
    agent_pivot = fold_agent_df.pivot(index="fold", columns="agent", values="count").fillna(0).astype(int)
    display(agent_pivot.iloc[:, :20])

    models = {}
    cv_results = []

    def evaluate_view_with_constraints(view_name, preds_proba, y_val, meta_val):
        all_p, all_t = [], []
        num_games = len(y_val) // 5

        for i in range(num_games):
            s, e = i * 5, (i + 1) * 5
            if view_name == "SEER":
                res = assign_roles_for_seer_by_divination(
                    preds_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                    meta_val['div_result1'][s:e], meta_val['div_id1'][s:e],
                    meta_val['div_result2'][s:e], meta_val['div_id2'][s:e],
                    meta_val['exec_id'][s:e], meta_val['attack_id'][s:e],
                    RUN_OPTIONS["day2_flag"]
                )
                for k in ["black", "white"]:
                    if res[k][0].size > 0:
                        all_p.extend(res[k][0])
                        all_t.extend(res[k][1])
            else:
                p, t = assign_roles_for_non_seer(
                    preds_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                    view_name, meta_val['exec_id'][s:e], meta_val['attack_id'][s:e],
                    RUN_OPTIONS["day2_flag"]
                )
                if p.size > 0:
                    all_p.extend(p)
                    all_t.extend(t)

        if len(all_t) == 0:
            return 0.0, 0

        target_label = 'POSSESSED' if view_name == 'WEREWOLF' else 'WEREWOLF'
        target_id = label_encoder.transform([target_label])[0]
        score = f1_score(all_t, all_p, labels=[target_id], average='macro', zero_division=0)
        return float(score), len(all_t)

    def train_model(view_name):
        print(f"\n  {'='*65}")
        print(f"  {view_name} 視点モデル: Optuna探索開始")
        print(f"  {'='*65}")

        def objective(trial):
            params = {
                'objective': 'multi:softprob',
                'num_class': len(classes),
                'eval_metric': 'mlogloss',
                'tree_method': 'hist',
                'random_state': 42,
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'n_estimators': trial.suggest_int('n_estimators', 150, 800),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            }

            fold_scores = []
            for fold, (train_idx, val_idx) in enumerate(split_indices, start=1):
                X_tr, X_val = X[train_idx], X[val_idx]
                y_tr, y_val = y[train_idx], y[val_idx]
                meta_val = {k: v[val_idx] for k, v in meta.items()}
                w_tr = np.array([weight_dict[label] for label in y_tr])

                model = xgb.XGBClassifier(**params)
                model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)

                preds_proba = model.predict_proba(X_val)
                score, _ = evaluate_view_with_constraints(view_name, preds_proba, y_val, meta_val)
                fold_scores.append(score)

                trial.report(np.mean(fold_scores), step=fold)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            return float(np.mean(fold_scores))

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_params = study.best_params.copy()
        best_params.update({
            'objective': 'multi:softprob',
            'num_class': len(classes),
            'eval_metric': 'mlogloss',
            'tree_method': 'hist',
            'random_state': 42,
        })

        print(f"\n    Best trial: {study.best_trial.number}")
        print(f"    Best CV score: {study.best_value:.4f}")
        print(f"    Best params: {study.best_params}")

        fold_scores = []
        fold_samples = []
        for fold, (train_idx, val_idx) in enumerate(split_indices, start=1):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            meta_val = {k: v[val_idx] for k, v in meta.items()}
            w_tr = np.array([weight_dict[label] for label in y_tr])

            model = xgb.XGBClassifier(**best_params)
            model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)

            preds_proba = model.predict_proba(X_val)
            fold_score, n_assigned = evaluate_view_with_constraints(view_name, preds_proba, y_val, meta_val)
            fold_scores.append(fold_score)
            fold_samples.append(n_assigned)
            print(f"    Fold {fold}/5: assigned={n_assigned}, F1={fold_score:.4f}")

        w_full = np.array([weight_dict[label] for label in y])
        final_model = xgb.XGBClassifier(**best_params)
        final_model.fit(X, y, sample_weight=w_full)
        models[view_name] = final_model

        mean_score = float(np.mean(fold_scores)) if fold_scores else 0.0
        print(f"    ✓ {view_name} モデル訓練完了 (mean F1={mean_score:.4f})")

        cv_results.append({
            'model': view_name,
            'mean_f1': mean_score,
            'fold_scores': fold_scores,
            'assigned_samples': fold_samples,
            'best_trial': int(study.best_trial.number),
            'best_params': study.best_params,
        })

        return final_model

    for view in ["SEER", "WEREWOLF", "VILLAGER", "POSSESSED"]:
        train_model(view)

    run_status["ok"] = True
    print(f"\n{'='*70}")
    print("✓ 全モデル訓練完了")
    print(f"{'='*70}")

    print("\nCV結果サマリー:")
    for result in cv_results:
        print(
            f"  {result['model']:<10} meanF1={result['mean_f1']:.4f} "
            f"best_trial={result['best_trial']}"
        )

except Exception as e:
    run_status["error"] = str(e)
    print(f"\nTraining failed: {e}")
    print(traceback.format_exc())
finally:
    run_status["elapsed_sec"] = round(time.time() - t0, 2)

print("\nRun status:")
print(json.dumps(run_status, ensure_ascii=False, indent=2))

2DAY START モデル訓練 (Optuna + Balanced Fold Split)

✓ Data prepared for training:
  Samples: 710
  Features: 420
  Classes: ['POSSESSED', 'SEER', 'VILLAGER', 'WEREWOLF']
  CV folds: 5
  Optuna trials: 30

Balanced fold summary:


,fold,n_samples,n_games
0,1,140,28
1,2,140,28
2,3,145,29
3,4,145,29
4,5,140,28


Fold x Dataset distribution:


dataset,2025spring,2025summer
fold,,
1,60,80
2,60,80
3,75,70
4,70,75
5,70,70


Fold x Role distribution:


...


    Best trial: 1
    Best CV score: 0.6017
    Best params: {'max_depth': 8, 'learning_rate': 0.010636066512540286, 'n_estimators': 781, 'min_child_weight': 17, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5909124836035503, 'gamma': 3.939402261362697e-07, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.00052821153945323}
    Fold 1/5: assigned=136, F1=0.6176
    Fold 2/5: assigned=136, F1=0.8235
    Fold 3/5: assigned=144, F1=0.5000
    Fold 4/5: assigned=148, F1=0.4865
    Fold 5/5: assigned=124, F1=0.5806


[I 2026-04-23 11:42:51,145] A new study created in memory with name: no-name-aa91ef06-ed80-49a4-98bb-343421824064


    ✓ VILLAGER モデル訓練完了 (mean F1=0.6017)

  POSSESSED 視点モデル: Optuna探索開始


[I 2026-04-23 11:43:02,465] Trial 0 finished with value: 0.5747252747252747 and parameters: {'max_depth': 5, 'learning_rate': 0.17254716573280354, 'n_estimators': 626, 'min_child_weight': 12, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 3.200866785899844e-08, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 0.5747252747252747.
[I 2026-04-23 11:43:17,967] Trial 1 finished with value: 0.5936813186813187 and parameters: {'max_depth': 8, 'learning_rate': 0.010636066512540286, 'n_estimators': 781, 'min_child_weight': 17, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5909124836035503, 'gamma': 3.939402261362697e-07, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.00052821153945323}. Best is trial 1 with value: 0.5936813186813187.
[I 2026-04-23 11:43:30,887] Trial 2 finished with value: 0.5678571428571428 and parameters: {'max_depth': 6, 'learning_rate': 0.023927528765580644, 'n_estimators': 548


    Best trial: 25
    Best CV score: 0.6198
    Best params: {'max_depth': 6, 'learning_rate': 0.018628257819241104, 'n_estimators': 720, 'min_child_weight': 18, 'subsample': 0.6271431188735435, 'colsample_bytree': 0.7246959009207872, 'gamma': 1.320974523201291e-08, 'reg_alpha': 5.970826690615007e-05, 'reg_lambda': 1.613793123308478e-06}
    Fold 1/5: assigned=52, F1=0.8462
    Fold 2/5: assigned=56, F1=0.5714
    Fold 3/5: assigned=52, F1=0.5385
    Fold 4/5: assigned=64, F1=0.5000
    Fold 5/5: assigned=56, F1=0.6429
    ✓ POSSESSED モデル訓練完了 (mean F1=0.6198)

✓ 全モデル訓練完了

CV結果サマリー:
  SEER       meanF1=0.9485 best_trial=18
  WEREWOLF   meanF1=0.4712 best_trial=21
  VILLAGER   meanF1=0.6017 best_trial=1
  POSSESSED  meanF1=0.6198 best_trial=25

Run status:
{
  "ok": true,
  "error": null,
  "elapsed_sec": 987.63
}


In [52]:
# Foldごとの target_label サポート数を可視化（CV条件一致版）
print("="*80)
print("Target label support by fold (CV-consistent)")
print("="*80)

if "cv_results" not in globals() or not cv_results:
    print("cv_results がありません。先に学習セルを実行してください。")
elif "split_indices" not in globals() or not split_indices:
    print("split_indices がありません。先に学習セルを実行してください。")
else:
    rows = []

    # 学習セルで得た best_params を view ごとに参照
    best_params_by_view = {r["model"]: r.get("best_params", {}) for r in cv_results if isinstance(r, dict)}

    for view_name in ["SEER", "WEREWOLF", "VILLAGER", "POSSESSED"]:
        if view_name not in best_params_by_view:
            print(f"{view_name}: best_params が見つからないためスキップ")
            continue

        best_params = dict(best_params_by_view[view_name])
        best_params.update({
            "objective": "multi:softprob",
            "num_class": len(classes),
            "eval_metric": "mlogloss",
            "tree_method": "hist",
            "random_state": 42,
        })

        target_label = "POSSESSED" if view_name == "WEREWOLF" else "WEREWOLF"
        target_id = label_encoder.transform([target_label])[0]

        for fold, (train_idx, val_idx) in enumerate(split_indices, start=1):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            meta_val = {k: v[val_idx] for k, v in meta.items()}
            w_tr = np.array([weight_dict[label] for label in y_tr])

            # CVと同じ条件: foldごとに train で再学習して val を評価
            fold_model = xgb.XGBClassifier(**best_params)
            fold_model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)

            preds_proba = fold_model.predict_proba(X_val)

            all_p, all_t = [], []
            num_games = len(y_val) // 5

            for i in range(num_games):
                s, e = i * 5, (i + 1) * 5
                if view_name == "SEER":
                    res = assign_roles_for_seer_by_divination(
                        preds_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                        meta_val["div_result1"][s:e], meta_val["div_id1"][s:e],
                        meta_val["div_result2"][s:e], meta_val["div_id2"][s:e],
                        meta_val["exec_id"][s:e], meta_val["attack_id"][s:e],
                        RUN_OPTIONS["day2_flag"]
                    )
                    for k in ["black", "white"]:
                        if res[k][0].size > 0:
                            all_p.extend(res[k][0])
                            all_t.extend(res[k][1])
                else:
                    p, t = assign_roles_for_non_seer(
                        preds_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                        view_name, meta_val["exec_id"][s:e], meta_val["attack_id"][s:e],
                        RUN_OPTIONS["day2_flag"]
                    )
                    if p.size > 0:
                        all_p.extend(p)
                        all_t.extend(t)

            assigned_n = len(all_t)
            target_support = int(np.sum(np.array(all_t) == target_id)) if assigned_n > 0 else 0
            target_f1 = (
                float(f1_score(all_t, all_p, labels=[target_id], average="macro", zero_division=0))
                if assigned_n > 0 else 0.0
            )

            rows.append({
                "view": view_name,
                "target_label": target_label,
                "fold": fold,
                "assigned_n": int(assigned_n),
                "target_support": target_support,
                "target_f1": target_f1,
            })

    diag_df = pd.DataFrame(rows)
    display(diag_df)

    if not diag_df.empty:
        print("\nPivot: target_support")
        display(diag_df.pivot(index="fold", columns="view", values="target_support"))
        print("\nPivot: target_f1")
        display(diag_df.pivot(index="fold", columns="view", values="target_f1"))

Target label support by fold (CV-consistent)


,view,target_label,fold,assigned_n,target_support,target_f1
0,SEER,WEREWOLF,1,36,9,1.000000
1,SEER,WEREWOLF,2,48,12,1.000000
2,SEER,WEREWOLF,3,44,11,1.000000
3,SEER,WEREWOLF,4,24,6,1.000000
4,SEER,WEREWOLF,5,16,4,1.000000
5,WEREWOLF,POSSESSED,1,108,27,0.962963
6,WEREWOLF,POSSESSED,2,108,27,0.851852
7,WEREWOLF,POSSESSED,3,108,27,0.814815
8,WEREWOLF,POSSESSED,4,104,26,1.000000
9,WEREWOLF,POSSESSED,5,108,27,0.962963



Pivot: target_support


view,POSSESSED,SEER,VILLAGER,WEREWOLF
fold,,,,
1,11,9,36,27
2,12,12,32,27
3,14,11,30,27
4,12,6,36,26
5,15,4,36,27



Pivot: target_f1


view,POSSESSED,SEER,VILLAGER,WEREWOLF
fold,,,,
1,1.000000,1.0,0.972222,0.962963
2,0.833333,1.0,0.843750,0.851852
3,0.928571,1.0,0.933333,0.814815
4,0.750000,1.0,0.833333,1.000000
5,1.000000,1.0,0.861111,0.962963


In [53]:
# 追加診断: 視点ごとのCV平均と target support/F1 集計
print("="*80)
print("Quick diagnostic summary")
print("="*80)

if "cv_results" in globals() and cv_results:
    cv_df = pd.DataFrame([{
        "view": r.get("model"),
        "cv_mean_f1": r.get("mean_f1"),
        "fold_scores": r.get("fold_scores"),
    } for r in cv_results])
    display(cv_df[["view", "cv_mean_f1"]].sort_values("cv_mean_f1"))
else:
    print("cv_results がありません")

if "diag_df" in globals() and isinstance(diag_df, pd.DataFrame) and not diag_df.empty:
    display(
        diag_df.groupby(["view", "target_label"], as_index=False).agg(
            folds=("fold", "count"),
            mean_assigned_n=("assigned_n", "mean"),
            min_target_support=("target_support", "min"),
            max_target_support=("target_support", "max"),
            mean_target_f1=("target_f1", "mean"),
            std_target_f1=("target_f1", "std"),
        ).sort_values("mean_target_f1")
    )
else:
    print("diag_df がありません")

Quick diagnostic summary


,view,cv_mean_f1
2,VILLAGER,0.888750
3,POSSESSED,0.902381
1,WEREWOLF,0.918519
0,SEER,1.000000


,view,target_label,folds,mean_assigned_n,min_target_support,max_target_support,mean_target_f1,std_target_f1
2,VILLAGER,WEREWOLF,5,136.0,30,36,0.888750,0.060859
0,POSSESSED,WEREWOLF,5,51.2,11,15,0.902381,0.109239
3,WEREWOLF,POSSESSED,5,107.2,26,27,0.918519,0.080294
1,SEER,WEREWOLF,5,33.6,4,12,1.000000,0.000000


In [17]:
# 追加診断2: WEREWOLF視点のfold別詳細
if "diag_df" in globals() and isinstance(diag_df, pd.DataFrame) and not diag_df.empty:
    ww = diag_df[diag_df["view"] == "WEREWOLF"].sort_values("fold")
    display(ww[["fold", "assigned_n", "target_support", "target_f1"]])
    print("WEREWOLF target_f1 fold values:", ww["target_f1"].round(4).tolist())
else:
    print("diag_df がありません")

,fold,assigned_n,target_support,target_f1
5,1,104,26,0.423077
6,2,112,28,0.571429
7,3,104,26,0.307692
8,4,104,26,0.730769
9,5,112,28,0.321429


WEREWOLF target_f1 fold values: [0.4231, 0.5714, 0.3077, 0.7308, 0.3214]


In [54]:
# 追加表示: 各foldに含まれるファイル一覧
print("="*80)
print("Files in each fold")
print("="*80)

if "fold_id_per_row" not in globals():
    print("fold_id_per_row がありません。先に学習セルを実行してください。")
else:
    fold_file_df = pd.DataFrame({
        "fold": fold_id_per_row + 1,
        "dataset_tag": meta["dataset_tag"],
        "source_file": meta["source_file"],
    }).drop_duplicates().sort_values(["fold", "dataset_tag", "source_file"]).reset_index(drop=True)

    # foldごとの件数サマリー
    fold_summary = fold_file_df.groupby("fold", as_index=False).agg(
        n_files=("source_file", "nunique"),
        n_datasets=("dataset_tag", "nunique"),
    )
    print("\n[Summary]")
    display(fold_summary)

    # fold別ファイル一覧
    for f in sorted(fold_file_df["fold"].unique()):
        sub = fold_file_df[fold_file_df["fold"] == f].copy()
        print(f"\n[Fold {f}] files={len(sub)}")
        display(sub[["dataset_tag", "source_file"]].reset_index(drop=True))

Files in each fold

[Summary]


,fold,n_files,n_datasets
0,1,27,2
1,2,25,2
2,3,26,2
3,4,24,2
4,5,26,2



[Fold 1] files=27


,dataset_tag,source_file
0,all_feature_table_2025sp17_with_talks,taged_1746438728_CanisLupus_kanolab_sunamelli_...
1,all_feature_table_2025sp17_with_talks,taged_1746440914_CamelliaDragons_kanolab_mille...
2,all_feature_table_2025sp17_with_talks,taged_1746442087_CanisLupus_ai168wolf_kanolab_...
3,all_feature_table_2025sp17_with_talks,taged_1746469395_kanolab_osawalab_sunamelli_GA...
4,all_feature_table_2025sp17_with_talks,taged_1746473490_sunamelli_ai168wolf_CamelliaD...
5,all_feature_table_2025sp17_with_talks,taged_1746474995_sunamelli_kanolab_GAIT_ai168w...
6,all_feature_table_2025sp17_with_talks,taged_1746528573_ai168wolf_GPTaku_CanisLupus_o...
7,all_feature_table_2025sp17_with_talks,taged_1746529637_osawalab_ai168wolf_GPTaku_kan...
8,all_feature_table_2025sp17_with_talks,taged_1746533146_kanolab_osawalab_sunamelli_GP...
9,all_feature_table_2025sp17_with_talks,taged_1746535428_ai168wolf_sunamelli_mille_osa...



[Fold 2] files=27


...

,dataset_tag,source_file
0,all_feature_table_2025sp17_with_talks,taged_1746429865_osawalab_ai168wolf_CamelliaDr...
1,all_feature_table_2025sp17_with_talks,taged_1746461224_mille_kanolab_osawalab_Camell...
2,all_feature_table_2025sp17_with_talks,taged_1746461792_CanisLupus_mille_GAIT_osawala...
3,all_feature_table_2025sp17_with_talks,taged_1746463069_GAIT_ai168wolf_CanisLupus_Cam...
4,all_feature_table_2025sp17_with_talks,taged_1746482677_CamelliaDragons_mille_kanolab...
5,all_feature_table_2025sp17_with_talks,taged_1746543128_mille_CanisLupus_osawalab_sun...
6,all_feature_table_2025sp17_with_talks,taged_1746547266_CamelliaDragons_mille_CanisLu...
7,all_feature_table_2025sp17_with_talks,taged_1746556910_CanisLupus_ai168wolf_osawalab...
8,all_feature_table_2025sp17_with_talks,taged_1746621585_GPTaku_kanolab_GAIT_CamelliaD...
9,all_feature_table_2025sp17_with_talks,taged_1746634155_GPTaku_kanolab_CanisLupus_ai1...



[Fold 4] files=26


,dataset_tag,source_file
0,all_feature_table_2025sp17_with_talks,taged_1746431630_kanolab_mille_ai168wolf_osawa...
1,all_feature_table_2025sp17_with_talks,taged_1746432378_mille_ai168wolf_osawalab_Came...
2,all_feature_table_2025sp17_with_talks,taged_1746441587_kanolab_CanisLupus_CamelliaDr...
3,all_feature_table_2025sp17_with_talks,taged_1746472566_kanolab_sunamelli_osawalab_mi...
4,all_feature_table_2025sp17_with_talks,taged_1746480219_kanolab_mille_CanisLupus_osaw...
5,all_feature_table_2025sp17_with_talks,taged_1746485144_ai168wolf_mille_CanisLupus_os...
6,all_feature_table_2025sp17_with_talks,taged_1746529140_CamelliaDragons_sunamelli_osa...
7,all_feature_table_2025sp17_with_talks,taged_1746531959_ai168wolf_osawalab_CamelliaDr...
8,all_feature_table_2025sp17_with_talks,taged_1746547899_GPTaku_ai168wolf_sunamelli_os...
9,all_feature_table_2025sp17_with_talks,taged_1746548423_sunamelli_CamelliaDragons_Can...



[Fold 5] files=27


,dataset_tag,source_file
0,all_feature_table_2025sp17_with_talks,taged_1746433451_kanolab_ai168wolf_sunamelli_o...
1,all_feature_table_2025sp17_with_talks,taged_1746439413_mille_CamelliaDragons_kanolab...
2,all_feature_table_2025sp17_with_talks,taged_1746440342_GAIT_sunamelli_osawalab_ai168...
3,all_feature_table_2025sp17_with_talks,taged_1746453871_CanisLupus_kanolab_sunamelli_...
4,all_feature_table_2025sp17_with_talks,taged_1746464608_sunamelli_CamelliaDragons_osa...
5,all_feature_table_2025sp17_with_talks,taged_1746471375_kanolab_GAIT_sunamelli_osawal...
6,all_feature_table_2025sp17_with_talks,taged_1746527448_GPTaku_sunamelli_ai168wolf_os...
7,all_feature_table_2025sp17_with_talks,taged_1746528097_CamelliaDragons_kanolab_GPTak...
8,all_feature_table_2025sp17_with_talks,taged_1746530082_GPTaku_CanisLupus_ai168wolf_m...
9,all_feature_table_2025sp17_with_talks,taged_1746598333_GAIT_osawalab_ai168wolf_mille...


## 7. Fold別・モデル別の詳細評価

各フォールドで役職割り当てロジックを使用した予測結果を詳細に分析します。

In [55]:
from sklearn.metrics import ConfusionMatrixDisplay

print("="*80)
print("2DAY START: Fold別・視点別の詳細評価")
print("="*80)

if not models:
    print("❌ モデルが訓練されていません")
else:
    # 学習セルで作成した balanced split を優先利用
    if 'split_indices' in globals() and split_indices:
        eval_splits = split_indices
        print("Using balanced split_indices from training cell.")
    else:
        sgkf = StratifiedGroupKFold(n_splits=5)
        eval_splits = list(sgkf.split(X, y, meta['source_file']))
        print("Using fallback StratifiedGroupKFold split.")

    for target_view in models.keys():
        print(f"\n{'='*80}")
        print(f"=== {target_view} 視点モデルの詳細評価 ===")
        print(f"{'='*80}")

        model = models[target_view]
        all_y_true_assigned = []
        all_y_pred_assigned = []

        for fold_idx, (train_idx, val_idx) in enumerate(eval_splits, start=1):
            X_val = X[val_idx]
            y_val = y[val_idx]
            meta_val = {k: v[val_idx] for k, v in meta.items()}

            y_pred_proba = model.predict_proba(X_val)
            num_games = len(y_val) // 5

            for i in range(num_games):
                s, e = i * 5, (i + 1) * 5

                if target_view == "SEER":
                    res = assign_roles_for_seer_by_divination(
                        y_pred_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                        meta_val['div_result1'][s:e], meta_val['div_id1'][s:e],
                        meta_val['div_result2'][s:e], meta_val['div_id2'][s:e],
                        meta_val['exec_id'][s:e], meta_val['attack_id'][s:e],
                        RUN_OPTIONS["day2_flag"]
                    )
                    for k in ['black', 'white']:
                        if res[k][0].size > 0:
                            all_y_pred_assigned.extend(res[k][0])
                            all_y_true_assigned.extend(res[k][1])
                else:
                    pred, true = assign_roles_for_non_seer(
                        y_pred_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                        target_view, meta_val['exec_id'][s:e], meta_val['attack_id'][s:e],
                        RUN_OPTIONS["day2_flag"]
                    )
                    if pred.size > 0:
                        all_y_pred_assigned.extend(pred)
                        all_y_true_assigned.extend(true)

        if len(all_y_true_assigned) > 0:
            print(f"\n評価サンプル数: {len(all_y_true_assigned)}")
            print("\n分類レポート:")
            print(classification_report(
                all_y_true_assigned,
                all_y_pred_assigned,
                labels=label_encoder.transform(class_names),
                target_names=class_names,
                zero_division=0,
            ))

            cm = confusion_matrix(
                all_y_true_assigned,
                all_y_pred_assigned,
                labels=label_encoder.transform(class_names),
            )
            cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
            print("\n混同行列:")
            print(cm_df)
        else:
            print("評価対象サンプルなし")

2DAY START: Fold別・視点別の詳細評価
Using balanced split_indices from training cell.

=== SEER 視点モデルの詳細評価 ===

評価サンプル数: 168

分類レポート:
              precision    recall  f1-score   support

   POSSESSED       1.00      1.00      1.00        42
        SEER       0.00      0.00      0.00         0
    VILLAGER       1.00      1.00      1.00        84
    WEREWOLF       1.00      1.00      1.00        42

    accuracy                           1.00       168
   macro avg       0.75      0.75      0.75       168
weighted avg       1.00      1.00      1.00       168


混同行列:
           POSSESSED  SEER  VILLAGER  WEREWOLF
POSSESSED         42     0         0         0
SEER               0     0         0         0
VILLAGER           0     0        84         0
WEREWOLF           0     0         0        42

=== WEREWOLF 視点モデルの詳細評価 ===

評価サンプル数: 536

分類レポート:
              precision    recall  f1-score   support

   POSSESSED       1.00      1.00      1.00       134
        SEER       1.00      1.00     

## 9. 結果の保存・エクスポート

モデル、特徴量重要度、評価結果をCSV・画像・joblib形式で保存します。

In [56]:
# 実験タグ管理: old/new 分割結果を混ぜないためのラベル
EXPERIMENT_TAG = globals().get("EXPERIMENT_TAG", "new_split")
SPLIT_VERSION = globals().get("SPLIT_VERSION", "dataset_plus_source")  # 例: source_only / dataset_plus_source
print("="*80)
print("Experiment tracking")
print("="*80)
print(f"EXPERIMENT_TAG : {EXPERIMENT_TAG}")
print(f"SPLIT_VERSION  : {SPLIT_VERSION}")
print("このタグ名で保存ファイルが分離されます。")

Experiment tracking
EXPERIMENT_TAG : new_split
SPLIT_VERSION  : dataset_plus_source
このタグ名で保存ファイルが分離されます。


In [58]:
print("="*80)
print("結果の保存")
print("="*80)

output_dir = PROJECT_ROOT / "notebooks" / "outputs" / "2day_start"
output_dir.mkdir(parents=True, exist_ok=True)

# 1. モデルの保存
if models:
    model_path = output_dir / "models_2day_start.joblib"
    joblib.dump(models, model_path)
    print(f"✓ Models saved: {model_path}")

# 2. 特徴量重要度の保存
if 'fi_df' in globals() and not fi_df.empty:
    fi_path = output_dir / "feature_importance_2day_start.csv"
    fi_df.to_csv(fi_path, index=False)
    print(f"✓ Feature importance saved: {fi_path}")

# 3. balanced fold分割情報の保存
if 'fold_id_per_row' in globals():
    fold_assign_df = pd.DataFrame({
        "source_file": meta["source_file"],
        "dataset_tag": meta["dataset_tag"] if "dataset_tag" in meta else "unknown",
        "agent_name": agent_names,
        "role": label_encoder.inverse_transform(y),
        "fold": fold_id_per_row + 1,
    })
    fold_assign_path = output_dir / "fold_assignments_balanced.csv"
    fold_assign_df.to_csv(fold_assign_path, index=False)
    print(f"✓ Fold assignments saved: {fold_assign_path}")

if 'fold_summary_df' in globals() and isinstance(fold_summary_df, pd.DataFrame):
    fold_summary_path = output_dir / "fold_split_basic_info.csv"
    fold_summary_df.to_csv(fold_summary_path, index=False)
    print(f"✓ Fold split summary saved: {fold_summary_path}")

if 'fold_role_df' in globals() and isinstance(fold_role_df, pd.DataFrame):
    fold_role_path = output_dir / "fold_role_distribution.csv"
    fold_role_df.to_csv(fold_role_path, index=False)
    print(f"✓ Fold role distribution saved: {fold_role_path}")

if 'fold_dataset_df' in globals() and isinstance(fold_dataset_df, pd.DataFrame):
    fold_dataset_path = output_dir / "fold_dataset_distribution.csv"
    fold_dataset_df.to_csv(fold_dataset_path, index=False)
    print(f"✓ Fold dataset distribution saved: {fold_dataset_path}")

if 'fold_agent_df' in globals() and isinstance(fold_agent_df, pd.DataFrame):
    fold_agent_path = output_dir / "fold_agent_distribution.csv"
    fold_agent_df.to_csv(fold_agent_path, index=False)
    print(f"✓ Fold agent distribution saved: {fold_agent_path}")

# 4. 設定とメタ情報の保存
metadata = {
    "generated_at": datetime.now().isoformat(),
    "notebook": "run_train_pipe_2day.ipynb",
    "data_paths": [str(p) for p in data_paths],
    "run_options": RUN_OPTIONS,
    "config": config,
    "models_trained": list(models.keys()) if models else [],
    "n_samples": len(X),
    "n_features": len(final_features),
    "roles": list(label_encoder.classes_),
    "cv_folds": 5,
    "split_strategy": "greedy_gamewise_balanced_by_dataset_and_agent",
}

metadata_path = output_dir / "metadata_2day_start.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"✓ Metadata saved: {metadata_path}")

# 5. 結果サマリー
print("\n" + "="*80)
print("2DAY START パイプライン完了")
print("="*80)
print(f"\n保存先: {output_dir}")
print(f"モデル数: {len(models) if models else 0}")
print(f"訓練サンプル数: {len(X)}")
print(f"特徴量数: {len(final_features)}")
print(f"実行時間: {run_status['elapsed_sec']} 秒")
print(f"ステータス: {'成功 ✓' if run_status['ok'] else '失敗 ✗'}")

saved_files = sorted(list(output_dir.glob("*")))
print(f"\n保存ファイル一覧:")
for f in saved_files:
    size_kb = f.stat().st_size / 1024 if f.is_file() else 0
    print(f"  - {f.name:<50} ({size_kb:>8.2f} KB)")

結果の保存
✓ Models saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\models_2day_start.joblib
✓ Fold assignments saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_assignments_balanced.csv
✓ Fold split summary saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_split_basic_info.csv
✓ Fold role distribution saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_role_distribution.csv
✓ Fold dataset distribution saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_dataset_distribution.csv
✓ Fold agent distribution saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_agent_distribution.csv
✓ Metadata saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\metadata_2day_start.json

2DAY START パイプライン完了

保存先: c:\User